In [1]:
import numpy as np

In [8]:
a = np.array([[4,3,8,4],[9,5,1,9],[2,7,6,2]])

In [35]:
def windows(grid):
    """always at least 3x3"""
    m, n = grid.shape
    for i in range(m-2):
        for j in range(n-2):
            yield grid[i:i+3, j:j+3]
def check_valid(grid):
    if np.any(grid>9):
        return False
    if np.any(grid<1):
        return False
    if len(np.unique(grid)) != 9:
        return False
    return True

def is_magic(grid):
    all_sum = np.sum(np.diag(grid))
    magic = False
    if all_sum == sum([grid[2,0], grid[1,1],grid[0,2]]):
        for i in range(3):
            if grid[i, :].sum() != all_sum:
                break
            if grid[:, i].sum() != all_sum:
                break
        else:
            magic = True
    return magic
        # diag_sum
        # row_sum
        # col_sum


In [36]:
sum([win[2,0], win[1,1],win[0,2]])

15

In [37]:
for win in windows(a):
    print(win, check_valid(win), is_magic(win))

[[4 3 8]
 [9 5 1]
 [2 7 6]] True True
[[3 8 4]
 [5 1 9]
 [7 6 2]] True False


In [43]:
np.array(win.shape) < 3

array([False, False])

In [17]:
for i in range(3-3):
    print(i)

In [127]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path().resolve().parent  # notebooks/ -> parent is project root
sys.path.append(str(PROJECT_ROOT))

In [115]:
import pandas as pd
from pandas import DataFrame
from pathlib import Path
from typing import Callable



def lazy(fn: Callable) -> property:
    """Decorator for caching properties after first load."""
    attr_name = f"_{fn.__name__}"
    @property
    def _lazy(self):
        """Gets attribute if it exists otherwise it sets the attribute"""
        if not hasattr(self, attr_name):
            setattr(self, attr_name, fn(self))
        return getattr(self, attr_name)

    return _lazy


class DataLoader:
    """
    `DataLoader` and basic formatter for NCAA M/W Basketball data
    """
    # File name constants
    REGULAR_SEASON_STATS_FILE = "RegularSeasonDetailedResults"
    TOURNAMENT_STATS_FILE = "NCAATourneyDetailedResults"
    TEAMS_FILE = "Teams"
    COACHES_FILE = "TeamCoaches"
    CONFERENCES_FILE = "TeamConferences"
    TOURNEY_SEEDS_FILE = "NCAATourneySeeds"
    TOURNEY_SLOTS_FILE = "NCAATourneySlots"

    def __init__(self, year: int=2025, men: bool=True) -> None:
        """
        Initializes `DataLoader` for a certain data pull year and gender

        Args:
            year: Year of the data pull
            men: boolean on if you want Mens (`True`) or Women's (`False`)

        Returns:
            `None`, but initilaizes dataloader object
        """
        self.year = year
        self.gender = "M" if men else "W"
        self.PROJECT_ROOT = Path().resolve().parent  # /src/data -> project root

    def _load(self, file_constant: str) -> DataFrame:
        """Internal helper to load a CSV file by name constant."""
        path = (
            self.PROJECT_ROOT
            / "data"
            / "raw"
            / f"march-machine-learning-mania-{self.year}"
            / f"{self.gender}{file_constant}.csv"
        )
        return pd.read_csv(path)

    # -----------------------------
    # Game-level data
    # -----------------------------
    @lazy
    def data(self):
        """load and concatenate regular and postseason data together."""
        # always concatenated from raw
        reg = self._load(self.REGULAR_SEASON_STATS_FILE)
        reg["Tournament"] = False

        tour = self._load(self.TOURNAMENT_STATS_FILE)
        tour["Tournament"] = True

        return pd.concat([reg, tour], ignore_index=True)

    # -----------------------------
    # Metadata
    # -----------------------------
    @lazy
    def teams(self):
        """Teams data such as IDs and Team names."""
        return self._load(self.TEAMS_FILE)

    @lazy
    def coaches(self):
        """Coaches data coach names and teams they coached for."""
        return self._load(self.COACHES_FILE)

    @lazy
    def conferences(self):
        """Conference information for teams"""
        return self._load(self.CONFERENCES_FILE)

    # -----------------------------
    # Tournament structure
    # -----------------------------
    @lazy
    def tourney_seeds(self):
        """Tournament seeding by year"""
        return self._load(self.TOURNEY_SEEDS_FILE)

    @lazy
    def tourney_slots(self):
        """Tournament slot information by year"""
        return self._load(self.TOURNEY_SLOTS_FILE)

    # -----------------------------
    # Preprocessing Pipeline
    # -----------------------------
    def preprocess(self) -> None:
        """Run all preprocessing steps and cache processed data."""
        data = self.data.copy()

        # Steps:
        steps = [
            self._create_game_id,
            self._possessions,
            self._minutes,
            self._create_team_game_stats,
            self._duplicate_and_flip_teams,
        ]

        for step in steps:
            print(step.__name__)
            data = step(data)

        # Cache processed data
        self._processed_data = data
#         return data

    # -----------------------------
    # Preprocessing Functions
    # -----------------------------

    def _possessions(self, data: DataFrame) -> DataFrame:
        """
        field goals attempted - offensive rebounds + turnovers + (0.475 x free throws attempted)
        """
        W = data["WFGA"] - data["WOR"] + data["WTO"] + (0.475 * data["WFTA"])
        L = data["LFGA"] - data["LOR"] + data["LTO"] + (0.475 * data["LFTA"])
        data["Possessions"] = ((W + L) // 2).astype(int)
        return data

    def _minutes(self, data: DataFrame) -> DataFrame:
        """calculate minutes in the game"""
        data["Minutes"] = 40 + data["NumOT"] * 5
        return data

    def _create_game_id(self, data: DataFrame) -> DataFrame:
        """GameID from season, DayNum, winning TeamID and Losing TeamID"""
        data["GameID"] = (
            data.Season.astype("str")
            + "_"
            + data.DayNum.astype(str).str.zfill(3)
            + "_"
            + data.WTeamID.astype("str")
            + "_"
            + data.LTeamID.astype("str")
        )
        return data

    def _create_team_game_stats(self, data: DataFrame) -> DataFrame:
        """rename columns from W/L to team/opponent."""
        data = data.rename(columns={"WLoc": "Loc"})
        winning_cols_mapping = {
            col: f"team_{col[1:]}" for col in data.columns if col.startswith("W")
        }
        losing_cols_mapping = {
            col: f"opponent_{col[1:]}"
            for col in data.columns
            if (col.startswith("L")) and (col != "Loc")
        }
        data = data.rename(columns=winning_cols_mapping)
        data = data.rename(columns=losing_cols_mapping)
        return data

    def _loser_loc_change(self, loc: str) -> str:
        """Changes the location to the opposite."""
        if loc == "H":
            return "A"
        elif loc == "A":
            return "H"
        else:
            return "N"

    def _duplicate_and_flip_teams(self, data: DataFrame) -> DataFrame:
        """flip team and opponent and concatenate to original."""
        data_flipped = data.copy()
        data_flipped = data_flipped.rename(
            columns={
                col: col.replace("team_", "opponent_")
                if col.startswith("team_")
                else col.replace("opponent_", "team_")
                for col in data_flipped.columns
                if (col.startswith("team_")) | (col.startswith("opponent_"))
            }
        )

        data_flipped.Loc = data_flipped.Loc.apply(self._loser_loc_change)
        return pd.concat([data, data_flipped], ignore_index=True)

    # Always run processing pipeline
    @property
    def processed_data(self) -> DataFrame:
        """processed and formatted games data."""
        if not hasattr(self, "_processed_data"):
            self.preprocess()
        return self._processed_data

    @property
    def regular_season_data(self) -> DataFrame:
        """processed and formatted regular season games data."""
        return self.processed_data.loc[~self.processed_data["Tournament"]].copy()

    @property
    def tournament_data(self) -> DataFrame:
        """processed and formatted tournament games data."""
        return self.processed_data.loc[self.processed_data["Tournament"]].copy()

    def write_processed_data(self) -> None:
        """pWrite data to interim table"""
        path = (
            self.PROJECT_ROOT
            / "data"
            / "interim"
            / f"{self.gender}{self.year}_formatted_data.csv"
        )
        self.processed_data.to_csv(path, index=False)
        print(f"Wrote data to: {path}")


print(DataLoader(2025).processed_data.head(5))


_create_game_id
_possessions
_minutes
_create_team_game_stats
_duplicate_and_flip_teams
   Season  DayNum  team_TeamID  team_Score  opponent_TeamID  opponent_Score  \
0    2003      10         1104          68             1328              62   
1    2003      10         1272          70             1393              63   
2    2003      11         1266          73             1437              61   
3    2003      11         1296          56             1457              50   
4    2003      11         1400          77             1208              71   

  Loc  NumOT  team_FGM  team_FGA  ...  opponent_DR  opponent_Ast  opponent_TO  \
0   N      0        27        58  ...           22             8           18   
1   N      0        26        62  ...           25             7           12   
2   N      0        24        58  ...           22             9           12   
3   N      0        18        38  ...           20             9           19   
4   N      0        30        61

In [133]:
"""Data Processing Functions and Logic"""
from pathlib import Path
from typing import Callable
import pandas as pd
from pandas import DataFrame


def lazy(func: Callable) -> property:
    """Decorator for caching properties after first load."""
    attr_name = f"_{func.__name__}"

    @property
    def _lazy(self):
        """Gets attribute if it exists otherwise it sets the attribute"""
        if not hasattr(self, attr_name):
            setattr(self, attr_name, func(self))
        return getattr(self, attr_name)

    return _lazy


class DataLoader:
    """
    `DataLoader` and basic formatter for NCAA M/W Basketball data
    """

    # File name constants
    REGULAR_SEASON_STATS_FILE = "RegularSeasonDetailedResults"
    TOURNAMENT_STATS_FILE = "NCAATourneyDetailedResults"
    TEAMS_FILE = "Teams"
    COACHES_FILE = "TeamCoaches"
    CONFERENCES_FILE = "TeamConferences"
    TOURNEY_SEEDS_FILE = "NCAATourneySeeds"
    TOURNEY_SLOTS_FILE = "NCAATourneySlots"


    def __init__(self, year: int = 2025, men: bool = True) -> None:
        """
        Initializes `DataLoader` for a certain data pull year and gender

        Args:
            year: Year of the data pull
            men: boolean on if you want Mens (`True`) or Women's (`False`)

        Returns:
            `None`, but initilaizes dataloader object
        """
        self.year = year
        self.gender = "M" if men else "W"
        self.project_root = Path().resolve().parent

    def _load(self, file_constant: str) -> DataFrame:
        """Internal helper to load a CSV file by name constant."""
        path = (
            self.project_root
            / "data"
            / "raw"
            / f"march-machine-learning-mania-{self.year}"
            / f"{self.gender}{file_constant}.csv"
        )
        return pd.read_csv(path)

    # -----------------------------
    # Game-level data
    # -----------------------------
    @lazy
    def data(self):
        """load and concatenate regular and postseason data together."""
        # always concatenated from raw
        reg = self._load(self.REGULAR_SEASON_STATS_FILE)
        reg["Tournament"] = False

        tour = self._load(self.TOURNAMENT_STATS_FILE)
        tour["Tournament"] = True

        return pd.concat([reg, tour], ignore_index=True)

    # -----------------------------
    # Metadata
    # -----------------------------
    @lazy
    def teams(self):
        """Teams data such as IDs and Team names."""
        return self._load(self.TEAMS_FILE)

    @lazy
    def coaches(self):
        """Coaches data coach names and teams they coached for."""
        return self._load(self.COACHES_FILE)

    @lazy
    def conferences(self):
        """Conference information for teams"""
        return self._load(self.CONFERENCES_FILE)

    # -----------------------------
    # Tournament structure
    # -----------------------------
    @lazy
    def tourney_seeds(self):
        """Tournament seeding by year"""
        return self._load(self.TOURNEY_SEEDS_FILE)

    @lazy
    def tourney_slots(self):
        """Tournament slot information by year"""
        return self._load(self.TOURNEY_SLOTS_FILE)

    # -----------------------------
    # Preprocessing Pipeline
    # -----------------------------
    
    
    
    def preprocess(self) -> DataFrame:
        """Run all preprocessing steps and cache processed data."""
        data = self.data.copy()

        # Steps:
        steps = [
            self._create_game_id,
            self._possessions,
            self._minutes,
            self._create_team_game_stats,
            self._duplicate_and_flip_teams,
            self._merge_coaches,
        ]

        for step in steps:
            print(step.__name__)
            data = step(data)
        
        return data

    # -----------------------------
    # Preprocessing Functions
    # -----------------------------

    def _possessions(self, data: DataFrame) -> DataFrame:
        """
        field goals attempted - offensive rebounds + turnovers + (0.475 x free throws attempted)
        """
        winning_team_poss = data["WFGA"] - data["WOR"] + data["WTO"] + (0.475 * data["WFTA"])
        losing_team_poss = data["LFGA"] - data["LOR"] + data["LTO"] + (0.475 * data["LFTA"])
        data["Possessions"] = ((winning_team_poss + losing_team_poss) // 2).astype(int)
        return data

    def _minutes(self, data: DataFrame) -> DataFrame:
        """calculate minutes in the game"""
        data["Minutes"] = 40 + data["NumOT"] * 5
        return data

    def _create_game_id(self, data: DataFrame) -> DataFrame:
        """GameID from season, DayNum, winning TeamID and Losing TeamID"""
        data["GameID"] = (
            data.Season.astype("str")
            + "_"
            + data.DayNum.astype(str).str.zfill(3)
            + "_"
            + data.WTeamID.astype("str")
            + "_"
            + data.LTeamID.astype("str")
        )
        return data

    def _create_team_game_stats(self, data: DataFrame) -> DataFrame:
        """rename columns from W/L to team/opponent."""
        data = data.rename(columns={"WLoc": "Loc"})
        winning_cols_mapping = {
            col: f"team_{col[1:]}" for col in data.columns if col.startswith("W")
        }
        losing_cols_mapping = {
            col: f"opponent_{col[1:]}"
            for col in data.columns
            if (col.startswith("L")) and (col != "Loc")
        }
        data = data.rename(columns=winning_cols_mapping)
        data = data.rename(columns=losing_cols_mapping)
        return data

    def _loser_loc_change(self, loc: str) -> str:
        """Changes the location to the opposite."""
        if loc == "H":
            return "A"
        if loc == "A":
            return "H"
        return "N"

    def _duplicate_and_flip_teams(self, data: DataFrame) -> DataFrame:
        """flip team and opponent and concatenate to original."""
        data_flipped = data.copy()
        data_flipped = data_flipped.rename(
            columns={
                col: col.replace("team_", "opponent_")
                if col.startswith("team_")
                else col.replace("opponent_", "team_")
                for col in data_flipped.columns
                if (col.startswith("team_")) | (col.startswith("opponent_"))
            }
        )

        data_flipped.Loc = data_flipped.Loc.apply(self._loser_loc_change)
        return pd.concat([data, data_flipped], ignore_index=True)
    
    def _merge_coaches(self, df: DataFrame) -> DataFrame:
        for team in ['team', 'opponent']:
            df = (
                df.merge(self.coaches,
                          left_on=["Season", f"{team}_TeamID"],
                          right_on=["Season", "TeamID"],
                          how="left")
            )

            # Filter where DayNum falls within the range
            df = df[
                (df["DayNum"] >= df["FirstDayNum"]) &
                (df["DayNum"] <= df["LastDayNum"])
            ]

            df = df.rename(columns={"CoachName": f'{team}_CoachName'})
            df = df.drop(columns=['FirstDayNum', 'LastDayNum', 'TeamID'])
        
        return df
    
    @lazy
    def processed_data(self):
        """processed and formatted games data."""
        return self.preprocess()

    @property
    def regular_season_data(self) -> DataFrame:
        """processed and formatted regular season games data."""
        return self.processed_data.loc[~self.processed_data["Tournament"]].copy()

    @property
    def tournament_data(self) -> DataFrame:
        """processed and formatted tournament games data."""
        return self.processed_data.loc[self.processed_data["Tournament"]].copy()

    def write_processed_data(self) -> None:
        """pWrite data to interim table"""
        path = (
            self.project_root
            / "data"
            / "interim"
            / f"{self.gender}{self.year}_formatted_data.csv"
        )
        self.processed_data.to_csv(path, index=False)
        print(f"Wrote data to: {path}")


print(DataLoader(2025).processed_data.head(5))


_create_game_id
_possessions
_minutes
_create_team_game_stats
_duplicate_and_flip_teams
_merge_coaches
   Season  DayNum  team_TeamID  team_Score  opponent_TeamID  opponent_Score  \
0    2003      10         1104          68             1328              62   
1    2003      10         1272          70             1393              63   
2    2003      11         1266          73             1437              61   
3    2003      11         1296          56             1457              50   
4    2003      11         1400          77             1208              71   

  Loc  NumOT  team_FGM  team_FGA  ...  opponent_TO  opponent_Stl  \
0   N      0        27        58  ...           18             9   
1   N      0        26        62  ...           12             8   
2   N      0        24        58  ...           12             2   
3   N      0        18        38  ...           19             4   
4   N      0        30        61  ...           10             7   

   opponent_B

In [134]:
m2025 = DataLoader(year=2025, men=True)

In [136]:
m2025.tourney_seeds

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
2621,2025,Z12,1161
2622,2025,Z13,1213
2623,2025,Z14,1423
2624,2025,Z15,1303
